# 컨텍스트 전처리 ① `applyToolResultBudget` 흉내내기 — 직전 사이클 도구결과 오프로드 (GPT Responses API)

Claude Code의 흐름은 **3계층**입니다 — 대화 턴(상위) ⊃ **ReAct 사이클(중위)** ⊃ 10단계 파이프라인(하위).
**컨텍스트 전처리는 중위 사이클마다 맨 위에서 딱 1번**, 모델을 부르기 직전에 돌며 **직전 사이클이 뱉은 도구결과 묶음**을 정리합니다.

전처리는 5단계인데, 이 노트북은 그중 **① `applyToolResultBudget`** 하나만 이식합니다.

| 단계 | 이름 | 보는 범위 | 하는 일 |
|---|---|---|---|
| **①** | **applyToolResultBudget** | **직전 1턴 묶음** | **묶음 총합이 200,000자 초과면 큰 결과를 디스크로 오프로드 + 미리보기/경로만 잔류** |
| ② | snipCompact | 누적 | 토큰 확보용 trimming |
| ③ | microcompact | 전체 대화 | 오래된 도구결과(최근 N개 제외) 내용 비움 |
| ④ | contextCollapse | 누적 | read/search 그룹 접기 |
| ⑤ | autocompact | 전체 대화 | 토큰 임계 초과 시 대화 전체를 인수인계 문서로 교체 |

## 이 노트북의 변형 — 200KB 게이트 대신 "Edit 결과 무조건"

실제 ①은 **묶음 총합 > 200,000자**일 때만 발동하는데, 테스트에서 200KB를 넘기기가 어렵습니다.
그래서 여기서는 **크기 조건 대신, 직전 사이클 묶음 중 `edit_file` 도구 결과만 무조건** 오프로드합니다.
나머지 메커니즘(전문은 오프로드, 컨텍스트엔 미리보기+포인터만)은 원본 그대로입니다.

- **문서화 = 변수 저장**: 실제 CC는 `~/.claude/projects/<proj>/<session>/tool-results/<id>.txt`에 전문을 씁니다.
  여기서는 테스트용이라 **노트북 변수(dict) `doc_store`** 에 `mem://edit-docs/<call_id>.txt → 전문`으로 저장합니다.
- **미리보기 크기**: 실제 `PREVIEW_SIZE_BYTES = 2000`. 데모에서 '미리보기 ≠ 전문' 대비가 눈에 보이도록 **400B**로 줄였습니다.

> 원문: `cc_agent_bible/md_group/cc-context-preprocessing-timing.md` · `대용량-결과-디스크-복구.md` · `도구결과-생애-읽기창.md`

In [ ]:
import json

from cc_tools import (
    FS,                 # cc_tools 내부 목 FS: path -> {"content", "mtime"}
    reset_fs,           # orderhub 40파일로 초기화
    get_client,         # load_dotenv() + OpenAI() (지연 생성)
    MODEL,              # "gpt-5-nano"
    BaseSession,        # Responses API ReAct 루프 골격
    SOFT_TOOLS,         # 소프트 도구 스키마 6종 (여기선 3개만 골라 씀)
    soft_read_file,     # read (오타/재읽기 넛지 켜짐, 게이트 없음)
    edit_file,          # 정확일치 교체 (state 없음 = 읽기강제 게이트 없음)
    glob_files,         # 패턴 매칭
    build_system_prompt,
)

n = reset_fs()
print(f"seeded {n} files (orderhub 공통 목 코드베이스)")
print("MODEL =", MODEL)

seeded 40 files (orderhub 공통 목 코드베이스)
MODEL = gpt-5-nano


## 1. 도구 무대 — `cc_tools`의 read/edit/glob 재사용

`cc_tool_sequence_soft_rules` 노트북이 쓰는 소프트 도구 한 벌을 그대로 가져와 **읽기·편집·글롭 3개만** 싣습니다.
편집은 `edit_file` — `old_string`이 파일 원문과 정확히 일치해야 하므로, 모델은 **먼저 읽어서** old_string을 조달합니다(읽기가 자연히 앞섬).
이 실제 툴콜링이 만들어내는 `edit_file` 결과가 다음 사이클 맨 위 전처리의 먹잇감이 됩니다.

In [ ]:
# SOFT_TOOLS(6개)에서 이 데모에 필요한 3개만 — 세션 내내 동결(캐시 미스 0)
NB_TOOLS = [t for t in SOFT_TOOLS if t["name"] in ("read_file", "edit_file", "glob_files")]
NB_IMPLS = {
    "read_file": soft_read_file,
    "edit_file": edit_file,
    "glob_files": glob_files,
}
print("실린 도구:", [t["name"] for t in NB_TOOLS])

SYS_PROMPT = build_system_prompt(
    "요청받은 편집을 모두 마치면 추가 설명 없이 한 줄로 완료만 보고하세요."
)

실린 도구: ['edit_file', 'glob_files', 'read_file']


## 2. 전처리 ① 유틸 — 미리보기 + 문서화

`edit_file` 결과 하나를 **문서(전문)** 와 **미리보기(head)** 로 가릅니다.

- `build_edit_document(...)` — 편집을 문서화: 파일 경로 · before→after · 도구 결과 원문 · **편집 후 파일 전문 스냅샷**. (실제 CC라면 디스크로 갈 전문)
- `persisted_message(...)` — 컨텍스트에 남길 교체 메시지: `<persisted-output>` + 포인터 + `Preview (first 400B)`. 원본 CC 메시지 형식(`Full output saved to: …` / `Preview (first 2KB):`)의 축소 모방.

In [ ]:
BUDGET_CHARS = 200_000   # CC applyToolResultBudget 임계 — 이 데모는 크기 대신 'Edit 결과'를 무조건 트리거
PREVIEW_BYTES = 400      # 데모용. 실제 CC는 PREVIEW_SIZE_BYTES = 2000


def build_edit_document(call_id, meta, raw_output):
    """Edit 결과를 '문서화' — 실제 CC라면 tool-results/<id>.txt 로 갈 전문."""
    fp = meta["file_path"]
    snapshot = FS[fp]["content"] if fp in FS else "(파일 없음)"
    return (
        f"[EDIT 문서화 · call_id={call_id} · 사이클 {meta['round']}]\n"
        f"파일: {fp}\n"
        f"교체(before→after):\n"
        f"- old: {meta['old_string']!r}\n"
        f"+ new: {meta['new_string']!r}\n"
        f"도구 결과 원문: {raw_output}\n"
        f"── 편집 후 파일 전문 스냅샷 ──\n{snapshot}"
    )


def _preview(doc):
    head = doc[:PREVIEW_BYTES]
    return head + ("\n...(잘림 — 전문은 포인터로)" if len(doc) > PREVIEW_BYTES else "")


def persisted_message(pointer, doc):
    """컨텍스트에 잔류할 교체 메시지 — 원본 CC buildLargeToolResultMessage 축소판."""
    return (
        "<persisted-output>\n"
        f"Edit 결과를 문서로 이관했습니다(컨텍스트 예산 보호). 전문 저장: {pointer}\n\n"
        f"Preview (first {PREVIEW_BYTES}B):\n{_preview(doc)}\n"
        "</persisted-output>"
    )

## 3. `PreprocessSession` — 사이클 맨 위에 전처리 ①을 끼운 ReAct 루프

`BaseSession`의 루프를 펼쳐, 매 사이클을 **[전처리 ① → 모델(Reason) → 도구실행(Act)]** 순서로 돌립니다.

- 전처리는 **모델 호출 직전**, `self.prev_round_outputs`(= 직전 사이클이 남긴 도구결과 묶음)를 대상으로 1번.
- 사이클 1은 직전 묶음이 없어 **no-op** ("사이클1 전처리는 거의 no-op").
- `edit_file` 결과는 `function_call_output` dict를 **제자리 치환**합니다 — 그 dict가 이미 `input_list`에 들어 있으므로, 다음 모델 호출은 미리보기+포인터만 봅니다.
- `self.processed`로 이미 오프로드한 결과는 재처리하지 않습니다(멱등 — 원본 `flag: 'wx'` 대응).

In [ ]:
def _norm_item(item):
    """input_list 항목(dict 또는 SDK 객체)을 결정론적 JSON 문자열로 — 프리픽스 비교용."""
    obj = item if isinstance(item, dict) else item.model_dump()
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, default=str)


class PreprocessSession(BaseSession):
    """전처리 ① applyToolResultBudget(Edit 강제 변형)을 사이클 맨 위에 끼운 세션."""

    def __init__(self, model=None):
        reset_fs()  # FS는 세션 단위 초기화
        super().__init__(SYS_PROMPT, NB_TOOLS, NB_IMPLS, model=model)
        self.doc_store = {}           # 문서화 저장소: mem:// 포인터 -> 전문 (디스크 오프로드 흉내)
        self.call_tool_name = {}      # call_id -> 도구 이름 (edit 판별)
        self.edit_meta = {}           # call_id -> {file_path, old_string, new_string, round}
        self.prev_round_outputs = []  # 직전 사이클 묶음(참조) = 다음 전처리 대상
        self.processed = set()        # 이미 오프로드한 call_id (멱등)
        self.round_no = 0
        self.usage_log = []           # 사이클(=모델 요청)별 KV 캐시 계측
        self.sent_snapshots = []      # 각 요청이 실제 보낸 input_list 직렬화 스냅샷(프리픽스 검증용)
        self._pending_offload = 0     # 이번 사이클 전처리가 오프로드한 건수

    # ── 전처리 ① — 사이클 맨 위, 모델 호출 직전, 직전 사이클 묶음 대상 ──
    def _apply_edit_budget(self):
        self._pending_offload = 0
        fresh = self.prev_round_outputs
        total = sum(len(it["output"]) for it in fresh)
        targets = [it for it in fresh
                   if self.call_tool_name.get(it["call_id"]) == "edit_file"
                   and it["call_id"] not in self.processed]
        cmp = "＜" if total < BUDGET_CHARS else "≥"
        print(f"  ┌─ 전처리 ① applyToolResultBudget · 직전 묶음 {len(fresh)}건 "
              f"총 {total:,}자 {cmp} 임계 {BUDGET_CHARS:,}자")
        if not fresh:
            print("  └─ 직전 사이클 없음 → no-op\n")
            return
        if not targets:
            print("  └─ 묶음에 Edit 결과 없음 → no-op\n")
            return
        print("  │  (크기는 임계 미달이지만, 이 데모는 Edit 결과를 무조건 오프로드)")
        for it in targets:
            cid = it["call_id"]
            doc = build_edit_document(cid, self.edit_meta[cid], it["output"])
            pointer = f"mem://edit-docs/{cid}.txt"
            self.doc_store[pointer] = doc             # 문서화 = 변수 저장
            replaced = persisted_message(pointer, doc)
            print(f"  │  🗂  {self.edit_meta[cid]['file_path']}  "
                  f"전문 {len(doc):,}자 → 미리보기 {len(replaced):,}자  ·  {pointer}")
            it["output"] = replaced                   # input_list 안의 dict를 제자리 치환
            self.processed.add(cid)
        self._pending_offload = len(targets)
        print(f"  └─ Edit 결과 {len(targets)}건 오프로드 완료\n")

    def ask(self, question, max_rounds=8):
        print(f"💬 {question}\n")
        self.input_list.append({"role": "user", "content": question})
        for _ in range(max_rounds):
            self.round_no += 1
            print(f"═══════ 사이클 {self.round_no} ═══════")
            self._apply_edit_budget()                    # [맨 위] 전처리 ①
            # 이 요청이 실제 보내는 input을 그대로 직렬화해 스냅샷 (프리픽스 안정성 검증용)
            self.sent_snapshots.append([_norm_item(it) for it in self.input_list])
            response = get_client().responses.create(    # 모델(Reason)
                model=self.model, input=self.input_list, tools=self.tools,
                prompt_cache_key="ctx-preproc-demo")     # 캐시 라우팅 고정(서버 노이즈↓)
            u = response.usage
            cached = getattr(getattr(u, "input_tokens_details", None), "cached_tokens", 0) or 0
            self.usage_log.append({
                "round": self.round_no,
                "sent_items": len(self.sent_snapshots[-1]),
                "input_tokens": u.input_tokens,
                "cached_tokens": cached,
                "offloaded": self._pending_offload,
            })
            self.input_list += response.output
            calls = [it for it in response.output if it.type == "function_call"]
            if not calls:
                print(f"🤖 {response.output_text}\n")
                return response.output_text
            self.prev_round_outputs = []                 # 이번 사이클 묶음 (다음 전처리 대상)
            for call in calls:                           # 도구실행(Act) — 받은 순서대로 순차
                args = json.loads(call.arguments)
                _, output = self._execute(call.name, args)
                self.call_tool_name[call.call_id] = call.name
                if call.name == "edit_file":
                    self.edit_meta[call.call_id] = {
                        "file_path": args.get("file_path"),
                        "old_string": args.get("old_string"),
                        "new_string": args.get("new_string"),
                        "round": self.round_no,
                    }
                item = {"type": "function_call_output", "call_id": call.call_id, "output": output}
                self.input_list.append(item)
                self.prev_round_outputs.append(item)
                mark = "⛔" if output.startswith("ERROR") else "🔧"
                print(f"  {mark} {call.name}({json.dumps(args, ensure_ascii=False)[:70]})"
                      f" → {output.splitlines()[0][:80]}")
            print()
        print("⚠️ 최대 라운드 초과 — 데모 중단")
        return None

## 4. 데모 — 여러 파일 편집 → 다음 사이클 맨 위에서 Edit 결과 오프로드

`config.py`의 상수 2개와 `common.py`의 docstring을 고치는 작업입니다.
모델이 **읽기 → 편집**을 여러 사이클에 걸쳐 하면, 편집이 일어난 **다음 사이클 맨 위**에서 전처리 ①이 그 `edit_file` 결과들을 잡아 오프로드합니다.

In [ ]:
session = PreprocessSession()
session.ask(
    "src/app/config.py에서 TIMEOUT을 60으로, RETRY_LIMIT을 5로 각각 바꿔줘. "
    "그리고 src/app/utils/common.py의 clamp 함수 첫 줄에 한 줄짜리 docstring을 추가해줘."
)

💬 src/app/config.py에서 TIMEOUT을 60으로, RETRY_LIMIT을 5로 각각 바꿔줘. 그리고 src/app/utils/common.py의 clamp 함수 첫 줄에 한 줄짜리 docstring을 추가해줘.

═══════ 사이클 1 ═══════
  ┌─ 전처리 ① applyToolResultBudget · 직전 묶음 0건 총 0자 ＜ 임계 200,000자
  └─ 직전 사이클 없음 → no-op



  🔧 read_file({"file_path": "/project/src/app/config.py"}) →      1	import os
  🔧 read_file({"file_path": "/project/src/app/utils/common.py"}) →      1	# TODO: 이 함수는 나중에 numpy 로 대체

═══════ 사이클 2 ═══════
  ┌─ 전처리 ① applyToolResultBudget · 직전 묶음 2건 총 1,293자 ＜ 임계 200,000자
  └─ 묶음에 Edit 결과 없음 → no-op



  🔧 read_file({"file_path": "/project/src/app/config.py"}) → 마지막으로 읽은 이후 파일이 변하지 않았습니다. 이 대화의 앞선 read_file 결과 내용이 여전히 유효합니다 - 다시 읽지 말고 그것을 참조

═══════ 사이클 3 ═══════
  ┌─ 전처리 ① applyToolResultBudget · 직전 묶음 1건 총 84자 ＜ 임계 200,000자
  └─ 묶음에 Edit 결과 없음 → no-op



  🔧 read_file({"file_path": "/project/src/app/config.py", "offset": 1, "limit": 2000) →      1	import os

═══════ 사이클 4 ═══════
  ┌─ 전처리 ① applyToolResultBudget · 직전 묶음 1건 총 724자 ＜ 임계 200,000자
  └─ 묶음에 Edit 결과 없음 → no-op



  🔧 edit_file({"file_path": "/project/src/app/config.py", "old_string": "TIMEOUT = 3) → /project/src/app/config.py 파일이 수정되었습니다. 1곳을 교체했습니다.
  🔧 edit_file({"file_path": "/project/src/app/config.py", "old_string": "RETRY_LIMIT) → /project/src/app/config.py 파일이 수정되었습니다. 1곳을 교체했습니다.
  ⛔ edit_file({"file_path": "/project/src/app/utils/common.py", "old_string": "def c) → ERROR: 바꿀 문자열을 파일에서 찾지 못했습니다.

═══════ 사이클 5 ═══════
  ┌─ 전처리 ① applyToolResultBudget · 직전 묶음 3건 총 204자 ＜ 임계 200,000자
  │  (크기는 임계 미달이지만, 이 데모는 Edit 결과를 무조건 오프로드)
  │  🗂  /project/src/app/config.py  전문 827자 → 미리보기 571자  ·  mem://edit-docs/call_qxEj4yM35u2Viw9GomBflXa1.txt
  │  🗂  /project/src/app/config.py  전문 833자 → 미리보기 571자  ·  mem://edit-docs/call_RVX5v45MRLkhaRw3NXFpipPz.txt
  │  🗂  /project/src/app/utils/common.py  전문 922자 → 미리보기 571자  ·  mem://edit-docs/call_4adQCDwsrbfZjzglo4oyR7Xb.txt
  └─ Edit 결과 3건 오프로드 완료



  🔧 read_file({"file_path": "/project/src/app/utils/common.py"}) → 마지막으로 읽은 이후 파일이 변하지 않았습니다. 이 대화의 앞선 read_file 결과 내용이 여전히 유효합니다 - 다시 읽지 말고 그것을 참조

═══════ 사이클 6 ═══════
  ┌─ 전처리 ① applyToolResultBudget · 직전 묶음 1건 총 84자 ＜ 임계 200,000자
  └─ 묶음에 Edit 결과 없음 → no-op



  🔧 edit_file({"file_path": "/project/src/app/utils/common.py", "old_string": "def c) → /project/src/app/utils/common.py 파일이 수정되었습니다. 1곳을 교체했습니다.

═══════ 사이클 7 ═══════
  ┌─ 전처리 ① applyToolResultBudget · 직전 묶음 1건 총 57자 ＜ 임계 200,000자
  │  (크기는 임계 미달이지만, 이 데모는 Edit 결과를 무조건 오프로드)
  │  🗂  /project/src/app/utils/common.py  전문 856자 → 미리보기 571자  ·  mem://edit-docs/call_d1qvaHTf6hQJeu2zjcAs4mwB.txt
  └─ Edit 결과 1건 오프로드 완료



  ⛔ edit_file({"file_path": "/project/src/app/utils/common.py", "old_string": "def c) → ERROR: 바꿀 문자열을 파일에서 찾지 못했습니다.

═══════ 사이클 8 ═══════
  ┌─ 전처리 ① applyToolResultBudget · 직전 묶음 1건 총 102자 ＜ 임계 200,000자
  │  (크기는 임계 미달이지만, 이 데모는 Edit 결과를 무조건 오프로드)
  │  🗂  /project/src/app/utils/common.py  전문 993자 → 미리보기 571자  ·  mem://edit-docs/call_j023P7PXSt6bJ7Wd2ABPAGed.txt
  └─ Edit 결과 1건 오프로드 완료



🤖 완료



'완료'

## 5. 결과 확인 — 컨텍스트엔 미리보기+포인터만, 전문은 `doc_store`에

세션이 끝난 뒤 `input_list`를 보면, 오프로드된 `edit_file` 결과들은 `<persisted-output>` 미리보기로 바뀌어 있습니다.
전문은 전부 `doc_store`(변수)에 안전하게 있습니다 — 컨텍스트 예산은 미리보기 크기만 씁니다.

In [6]:
# 컨텍스트(input_list)에 남은 도구결과 중 오프로드(persisted)된 것만 추림
persisted = [it for it in session.input_list
             if isinstance(it, dict) and it.get("type") == "function_call_output"
             and it["output"].startswith("<persisted-output>")]
print(f"오프로드된 Edit 결과: {len(persisted)}건 / 문서 저장소 doc_store: {len(session.doc_store)}건\n")
for it in persisted:
    print(it["output"])
    print("─" * 64)

print("\n📦 doc_store 포인터 목록:")
for ptr, doc in session.doc_store.items():
    print(f"  {ptr}  (전문 {len(doc):,}자)")

오프로드된 Edit 결과: 5건 / 문서 저장소 doc_store: 5건

<persisted-output>
Edit 결과를 문서로 이관했습니다(컨텍스트 예산 보호). 전문 저장: mem://edit-docs/call_qxEj4yM35u2Viw9GomBflXa1.txt

Preview (first 400B):
[EDIT 문서화 · call_id=call_qxEj4yM35u2Viw9GomBflXa1 · 사이클 4]
파일: /project/src/app/config.py
교체(before→after):
- old: 'TIMEOUT = 30'
+ new: 'TIMEOUT = 60'
도구 결과 원문: /project/src/app/config.py 파일이 수정되었습니다. 1곳을 교체했습니다.
── 편집 후 파일 전문 스냅샷 ──
import os

DEBUG = True
TIMEOUT = 60
RETRY_LIMIT = 5
ALLOWED_HOSTS = ["localhost", "api.example.com"]


class Settings:
    # TODO: pydantic-settings 로 이전하고 이 수동 클래스
...(잘림 — 전문은 포인터로)
</persisted-output>
────────────────────────────────────────────────────────────────
<persisted-output>
Edit 결과를 문서로 이관했습니다(컨텍스트 예산 보호). 전문 저장: mem://edit-docs/call_RVX5v45MRLkhaRw3NXFpipPz.txt

Preview (first 400B):
[EDIT 문서화 · call_id=call_RVX5v45MRLkhaRw3NXFpipPz · 사이클 4]
파일: /project/src/app/config.py
교체(before→after):
- old: 'RETRY_LIMIT = 3'
+ new: 'RETRY_LIMIT = 5'
도구 결과 원문: /project/src/app/conf

## 6. 복구(recall) — 포인터로 전문 다시 꺼내기 (READ WINDOW)

미리보기에서 "더 봐야 할 신호"를 발견하면, 모델은 **창이 열린 그 자리에서** 포인터를 열어 전문을 꺼냅니다 —
실제 CC는 `Read(path, offset, limit)`, 여기서는 `doc_store[pointer]`. 창(READ WINDOW)이 닫히면(microcompact/autocompact) 경로째 사라져 못 읽습니다.

In [7]:
# 첫 문서를 포인터로 복구 — 전문 확인 (컨텍스트엔 미리보기만 있었지만 전문은 doc_store에 보존)
if session.doc_store:
    ptr = next(iter(session.doc_store))
    print(f"recall {ptr} →\n")
    print(session.doc_store[ptr])
else:
    print("(오프로드된 문서가 없습니다 — 편집이 한 사이클에 몰려 끝났을 수 있습니다)")

recall mem://edit-docs/call_qxEj4yM35u2Viw9GomBflXa1.txt →

[EDIT 문서화 · call_id=call_qxEj4yM35u2Viw9GomBflXa1 · 사이클 4]
파일: /project/src/app/config.py
교체(before→after):
- old: 'TIMEOUT = 30'
+ new: 'TIMEOUT = 60'
도구 결과 원문: /project/src/app/config.py 파일이 수정되었습니다. 1곳을 교체했습니다.
── 편집 후 파일 전문 스냅샷 ──
import os

DEBUG = True
TIMEOUT = 60
RETRY_LIMIT = 5
ALLOWED_HOSTS = ["localhost", "api.example.com"]


class Settings:
    # TODO: pydantic-settings 로 이전하고 이 수동 클래스는 제거
    DEBUG = DEBUG
    DATABASE_URL = os.getenv("DATABASE_URL", "postgresql+psycopg://localhost/orderhub")
    REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379/0")
    JWT_SECRET = os.getenv("JWT_SECRET", "change-me-in-production")
    JWT_EXPIRE_MINUTES = int(os.getenv("JWT_EXPIRE_MINUTES", "60"))
    PAYMENT_GATEWAY_URL = os.getenv("PAYMENT_GATEWAY_URL", "https://pay.example.com/v2")


settings = Settings()



## 7. KV 캐시 확인 — 전처리 오프로드가 프리픽스를 깼나? (ReAct 사이클 단위)

전처리 ①은 `function_call_output`을 **in-place로 치환**한다(`it["output"] = replaced`). 이미 캐시된 프리픽스를 바꾸면 그 지점부터 KV 캐시가 깨지므로, "정말 안 깨졌나"를 **사이클(=모델 요청)마다** 확인한다.

**안 깨지는 이유(핵심)**: 전처리는 **직전 사이클 묶음** — 바로 전 사이클이 만들었고 *아직 모델에 한 번도 안 보낸* fresh tool 출력 — 만 손댄다. 안정 프리픽스(시스템 프롬프트 + tools + 그 이전 사이클들)는 절대 건드리지 않는다. 그래서 오프로드가 일어난 사이클에서도 **직전 요청의 내용 전체가 현재 요청의 접두어로 그대로 남는다** → 캐시 프리픽스 무효화 없음.

두 각도로 검증한다:
- **실측 `cached_tokens`** (OpenAI usage): 요청이 커질수록 재사용 토큰이 유지·증가하면 프리픽스 생존. (프롬프트가 1024토큰 미만인 초기 사이클은 캐시 임계 미달이라 0일 수 있음 — 이는 '깨짐'이 아니라 '아직 캐시 대상 아님'. `prompt_cache_key` 고정으로 서버 라우팅 노이즈를 줄임.)
- **프리픽스 안정성**(결정론적): req N의 직렬화 전체가 req N+1의 접두어와 정확히 일치하는지. 오프로드 사이클에서도 불변이면 캐시 안전 확정.

> 원본 CC: microcompact는 캐시를 보존, autocompact만 프리픽스를 통째로 갈아 MISS. ①(예산 오프로드)은 fresh 묶음만 손대므로 microcompact처럼 캐시 보존 쪽이다.

In [8]:
# ── 사이클(=모델 요청)별 KV 캐시 계측 ──
print("사이클 | 보낸항목 | input_tok | cached_tok |  적중  | 오프로드")
print("─" * 60)
for r in session.usage_log:
    hit = "✅HIT" if r["cached_tokens"] > 0 else ("❄️COLD" if r["round"] == 1 else "· 미만")
    off = f"🗂×{r['offloaded']}" if r["offloaded"] else ""
    print(f"  {r['round']:>4} | {r['sent_items']:>7} | {r['input_tokens']:>8,} | "
          f"{r['cached_tokens']:>9,} | {hit:>5} | {off}")

# ── 프리픽스 안정성: 직전 요청 내용이 현재 요청의 '접두어'로 그대로 살아있나 ──
snaps = session.sent_snapshots
print("\n프리픽스 안정성 검증 (req N 전체가 req N+1의 접두어로 불변인가):")
all_ok = True
for i in range(len(snaps) - 1):
    prev, cur = snaps[i], snaps[i + 1]
    ok = cur[:len(prev)] == prev              # 직전 요청 전체가 현재 요청 접두어와 정확 일치
    all_ok &= ok
    tag = ""
    if session.usage_log[i + 1]["offloaded"]:
        tag = (f"   ← 이 사이클 맨 위에서 Edit {session.usage_log[i + 1]['offloaded']}건 "
               "in-place 오프로드 (그런데도 불변인 게 핵심)")
    print(f"  사이클 {i + 1}→{i + 2}: 직전 {len(prev):>2}개 항목 {'불변 ✓' if ok else '깨짐 ✗'}{tag}")

print("\n결론:",
      "전처리 ①이 '직전 사이클 묶음(아직 안 보낸 fresh 출력)'만 치환 → 안정 프리픽스 불변 → KV 캐시 안 깨짐 ✓"
      if all_ok else "프리픽스가 깨진 지점 존재 ✗")

사이클 | 보낸항목 | input_tok | cached_tok |  적중  | 오프로드
────────────────────────────────────────────────────────────
     1 |       2 |      636 |         0 | ❄️COLD | 
     2 |       7 |    1,548 |     1,024 |  ✅HIT | 
     3 |      10 |    1,766 |     1,664 |  ✅HIT | 
     4 |      13 |    2,817 |     2,432 |  ✅HIT | 
     5 |      20 |    5,052 |     4,096 |  ✅HIT | 🗂×3
     6 |      23 |    5,326 |     5,248 |  ✅HIT | 
     7 |      26 |    6,159 |     5,888 |  ✅HIT | 🗂×1
     8 |      29 |    7,941 |     6,656 |  ✅HIT | 🗂×1

프리픽스 안정성 검증 (req N 전체가 req N+1의 접두어로 불변인가):
  사이클 1→2: 직전  2개 항목 불변 ✓
  사이클 2→3: 직전  7개 항목 불변 ✓
  사이클 3→4: 직전 10개 항목 불변 ✓
  사이클 4→5: 직전 13개 항목 불변 ✓   ← 이 사이클 맨 위에서 Edit 3건 in-place 오프로드 (그런데도 불변인 게 핵심)
  사이클 5→6: 직전 20개 항목 불변 ✓
  사이클 6→7: 직전 23개 항목 불변 ✓   ← 이 사이클 맨 위에서 Edit 1건 in-place 오프로드 (그런데도 불변인 게 핵심)
  사이클 7→8: 직전 26개 항목 불변 ✓   ← 이 사이클 맨 위에서 Edit 1건 in-place 오프로드 (그런데도 불변인 게 핵심)

결론: 전처리 ①이 '직전 사이클 묶음(아직 안 보낸 fresh 출력)'만 치환 → 안정 프리픽스 불변 → KV 캐시 안 깨짐 ✓
